# Gemma 4 Stage 2 Translation Model Evaluation
### Bidirectional Translation: Algerian Darija (العامية الجزائرية) $\longleftrightarrow$ English

This notebook evaluates the Stage 2 fine-tuned model (`gemma4-darija-en-translation-qlora`):
- **Bidirectional Evaluation:**
  - **English $\to$ Algerian Darija (`en_to_ar`)**
  - **Algerian Darija $\to$ English (`ar_to_en`)**
- **Evaluation Dataset:**
  - Held-out test split from `algerian_translation_50k_cleaned.csv` ($5\%$ split with `seed=42`, matching Stage 2 training data separation).
- **Standard Translation Metrics:**
  - **BLEU / SacreBLEU:** Standard corpus and sentence-level n-gram precision.
  - **chrF++:** Character n-gram F-score with word order 2 (indispensable for morphologically rich and dialectal Arabic).
  - **BERTScore:** Semantic similarity via contextual multilingual embeddings.
  - **COMET:** Cross-lingual neural quality scoring with reference & source context (optional if installed).
  - **Brevity Penalty (BP) & Length Ratio:** Quality safeguards against truncation or repetitive loop hallucinations.
- **Analysis & Diagnostics:**
  - Score distribution histograms across both translation directions.
  - Best and worst translation qualitative sample inspection.
  - Export of predictions and metric summaries to CSV and JSON.

In [ ]:
# Environment and dependencies setup
# If running in a new environment, install required packages:
# !pip install -q sacrebleu bert-score evaluate unbabel-comet tabulate pandas matplotlib tqdm

import os
import sys
import gc
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# Setup MT Evaluation Metrics
import sacrebleu

# Availability flags
USE_BERTSCORE = True
USE_COMET = False  # Set to True if unbabel-comet is installed in your environment

try:
    import bert_score
    print("✓ bert_score is available.")
except ImportError:
    print("ℹ bert_score not found. To enable: pip install bert-score")
    USE_BERTSCORE = False

try:
    import comet
    print("✓ comet is available.")
except ImportError:
    print("ℹ comet not found. To enable: pip install unbabel-comet")
    USE_COMET = False


In [ ]:
# Load Model & Stage 2 Translation LoRA Adapter
model_id = "google/gemma-4-E2B"
adapter_path = "./gemma4-darija-en-translation-qlora"
compute_dtype = torch.float16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-padding for batched autoregressive generation

print(f"Loading 4-bit base model {model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map={"": 0} if torch.cuda.is_available() else None,
    dtype=compute_dtype,
)

print(f"Loading Stage 2 LoRA adapter from {adapter_path}...")
model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)
model.eval()
print("✓ Model ready for evaluation.")


In [ ]:
# Translation Prompt Templates matching Stage 2 fine-tuning exactly
EN_TO_AR_TEMPLATE = """ترجم الجملة التالية من الإنجليزية إلى الدارجة الجزائرية:

{sentence}

الترجمة: """

AR_TO_EN_TEMPLATE = """Translate the following Algerian Darija sentence to English:

{sentence}

Translation: """

def generate_translations_batch(sentences, direction="en_to_ar", batch_size=8, max_new_tokens=128):
    """
    Performs batched greedy generation for reproducible and fast evaluation.
    """
    if direction == "en_to_ar":
        prompts = [EN_TO_AR_TEMPLATE.format(sentence=s) for s in sentences]
    elif direction == "ar_to_en":
        prompts = [AR_TO_EN_TEMPLATE.format(sentence=s) for s in sentences]
    else:
        raise ValueError("Invalid direction. Choose 'en_to_ar' or 'ar_to_en'.")

    translations = []
    
    for i in tqdm(range(0, len(prompts), batch_size), desc=f"Evaluating ({direction})"):
        batch_prompts = prompts[i:i + batch_size]
        
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,        # Greedy decoding for consistent deterministic evaluation
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            
        for out in outputs:
            # Decode only the generated completion tokens beyond the prompt
            gen_tokens = out[inputs["input_ids"].shape[1]:]
            decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
            # Clean up trailing newlines
            decoded = decoded.split("\n\n")[0].strip()
            translations.append(decoded)
            
    return translations


In [ ]:
# Load Cleaned Translation Dataset & Extract 5% Test Split
csv_path = "algerian_translation_50k_cleaned.csv"
df_raw = pd.read_csv(csv_path).dropna(subset=["Arabic", "English"])

df_raw["Arabic"] = df_raw["Arabic"].astype(str).str.strip()
df_raw["English"] = df_raw["English"].astype(str).str.strip()
df_clean = df_raw[(df_raw["Arabic"] != "") & (df_raw["English"] != "")].reset_index(drop=True)

# Exact 5% held-out test split with seed=42
_, df_test = train_test_split(df_clean, test_size=0.05, random_state=42)
print(f"Total held-out test pairs available: {len(df_test)}")

# EVAL_SAMPLE_SIZE: You can evaluate on a subset (e.g. 500) for fast turnaround,
# or set to None to evaluate all ~2,540 test pairs.
EVAL_SAMPLE_SIZE = 500

if EVAL_SAMPLE_SIZE is not None and len(df_test) > EVAL_SAMPLE_SIZE:
    eval_df = df_test.sample(n=EVAL_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Selected representative test sample of: {len(eval_df)} pairs")
else:
    eval_df = df_test.reset_index(drop=True)
    print(f"Using complete test split: {len(eval_df)} pairs")

print("\nSample test pair:")
print("  Arabic  (Darija) :", eval_df.iloc[0]["Arabic"])
print("  English (Target) :", eval_df.iloc[0]["English"])


In [ ]:
# Direction 1: English -> Algerian Darija (en_to_ar)
sources_en = eval_df["English"].tolist()
references_ar = eval_df["Arabic"].tolist()

print(f"Generating translations for {len(sources_en)} sentences (English -> Darija)...")
preds_ar = generate_translations_batch(
    sources_en,
    direction="en_to_ar",
    batch_size=8,
    max_new_tokens=128
)
eval_df["pred_ar"] = preds_ar

print("\nSample Predictions (English -> Darija):")
for idx in range(min(5, len(eval_df))):
    print(f"[{idx+1}] EN Source : {eval_df.iloc[idx]['English']}")
    print(f"    AR Target : {eval_df.iloc[idx]['Arabic']}")
    print(f"    AR Pred   : {eval_df.iloc[idx]['pred_ar']}\n")


In [ ]:
# Direction 2: Algerian Darija -> English (ar_to_en)
sources_ar = eval_df["Arabic"].tolist()
references_en = eval_df["English"].tolist()

print(f"Generating translations for {len(sources_ar)} sentences (Darija -> English)...")
preds_en = generate_translations_batch(
    sources_ar,
    direction="ar_to_en",
    batch_size=8,
    max_new_tokens=128
)
eval_df["pred_en"] = preds_en

print("\nSample Predictions (Darija -> English):")
for idx in range(min(5, len(eval_df))):
    print(f"[{idx+1}] AR Source : {eval_df.iloc[idx]['Arabic']}")
    print(f"    EN Target : {eval_df.iloc[idx]['English']}")
    print(f"    EN Pred   : {eval_df.iloc[idx]['pred_en']}\n")


In [ ]:
# Metric Computation Functions
def compute_translation_metrics(hypotheses, references, sources=None, lang_pair="en-ar"):
    """
    Computes BLEU, chrF++, BERTScore, and COMET.
    """
    results = {}
    
    # 1. SacreBLEU
    tokenize_opt = "intl" if "ar" in lang_pair else "13a"
    bleu_calc = sacrebleu.corpus_bleu(hypotheses, [references], tokenize=tokenize_opt)
    results["BLEU"] = round(bleu_calc.score, 2)
    results["BLEU_bp"] = round(bleu_calc.bp, 4)
    results["sys_len"] = bleu_calc.sys_len
    results["ref_len"] = bleu_calc.ref_len
    results["len_ratio"] = round(bleu_calc.sys_len / max(1, bleu_calc.ref_len), 3)
    
    # 2. chrF++ (word_order=2 for chrF++)
    chrf_calc = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    results["chrF++"] = round(chrf_calc.score, 2)
    
    # Sentence-level chrF++
    sent_chrf = [
        sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score
        for hyp, ref in zip(hypotheses, references)
    ]
    results["sentence_chrF++"] = sent_chrf
    
    # 3. BERTScore
    if USE_BERTSCORE:
        try:
            from bert_score import score as bert_scorer
            target_lang = "ar" if lang_pair.endswith("ar") else "en"
            P, R, F1 = bert_scorer(
                hypotheses,
                references,
                lang=target_lang,
                verbose=False,
                device=model.device if torch.cuda.is_available() else "cpu"
            )
            results["BERTScore_Precision"] = round(P.mean().item() * 100, 2)
            results["BERTScore_Recall"] = round(R.mean().item() * 100, 2)
            results["BERTScore_F1"] = round(F1.mean().item() * 100, 2)
            results["sentence_BERTScore_F1"] = [score.item() * 100 for score in F1]
        except Exception as e:
            print(f"Notice: BERTScore skipped: {e}")
            
    # 4. COMET
    if USE_COMET and sources is not None:
        try:
            from comet import download_model, load_from_checkpoint
            comet_path = download_model("Unbabel/wmt22-comet-da")
            comet_model = load_from_checkpoint(comet_path)
            comet_data = [
                {"src": s, "mt": h, "ref": r}
                for s, h, r in zip(sources, hypotheses, references)
            ]
            comet_res = comet_model.predict(comet_data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
            results["COMET"] = round(comet_res.system_score * 100, 2)
            results["sentence_COMET"] = comet_res.scores
        except Exception as e:
            print(f"Notice: COMET skipped: {e}")
            
    return results


In [ ]:
# Run Metric Computations
print("=== Computing Metrics: English -> Algerian Darija ===")
metrics_en_to_ar = compute_translation_metrics(
    hypotheses=eval_df["pred_ar"].tolist(),
    references=eval_df["Arabic"].tolist(),
    sources=eval_df["English"].tolist(),
    lang_pair="en-ar"
)

print("\n=== Computing Metrics: Algerian Darija -> English ===")
metrics_ar_to_en = compute_translation_metrics(
    hypotheses=eval_df["pred_en"].tolist(),
    references=eval_df["English"].tolist(),
    sources=eval_df["Arabic"].tolist(),
    lang_pair="ar-en"
)


In [ ]:
# Build Comparison Summary Table
summary_data = {
    "Metric": [
        "BLEU (sacrebleu)",
        "chrF++",
        "Brevity Penalty (BP)",
        "Hypothesis / Reference Length Ratio"
    ],
    "English -> Darija (en_to_ar)": [
        metrics_en_to_ar.get("BLEU", "N/A"),
        metrics_en_to_ar.get("chrF++", "N/A"),
        metrics_en_to_ar.get("BLEU_bp", "N/A"),
        metrics_en_to_ar.get("len_ratio", "N/A"),
    ],
    "Darija -> English (ar_to_en)": [
        metrics_ar_to_en.get("BLEU", "N/A"),
        metrics_ar_to_en.get("chrF++", "N/A"),
        metrics_ar_to_en.get("BLEU_bp", "N/A"),
        metrics_ar_to_en.get("len_ratio", "N/A"),
    ]
}

if "BERTScore_F1" in metrics_en_to_ar:
    summary_data["Metric"].extend(["BERTScore F1 (%)", "BERTScore Precision (%)", "BERTScore Recall (%)"])
    summary_data["English -> Darija (en_to_ar)"].extend([
        metrics_en_to_ar.get("BERTScore_F1"),
        metrics_en_to_ar.get("BERTScore_Precision"),
        metrics_en_to_ar.get("BERTScore_Recall"),
    ])
    summary_data["Darija -> English (ar_to_en)"].extend([
        metrics_ar_to_en.get("BERTScore_F1"),
        metrics_ar_to_en.get("BERTScore_Precision"),
        metrics_ar_to_en.get("BERTScore_Recall"),
    ])

if "COMET" in metrics_en_to_ar:
    summary_data["Metric"].append("COMET Score")
    summary_data["English -> Darija (en_to_ar)"].append(metrics_en_to_ar.get("COMET"))
    summary_data["Darija -> English (ar_to_en)"].append(metrics_ar_to_en.get("COMET"))

summary_table = pd.DataFrame(summary_data)

print("\n" + "="*65)
print("       STAGE 2 BIDIRECTIONAL EVALUATION BENCHMARK RESULTS       ")
print("="*65)
print(summary_table.to_string(index=False))
print("="*65)


In [ ]:
# Visualizations: Score Distributions Across Directions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot chrF++ distribution
axes[0].hist(metrics_en_to_ar["sentence_chrF++"], bins=25, alpha=0.6, label="English -> Darija", color="#1f77b4", edgecolor="black")
axes[0].hist(metrics_ar_to_en["sentence_chrF++"], bins=25, alpha=0.6, label="Darija -> English", color="#2ca02c", edgecolor="black")
axes[0].set_title("Sentence-Level chrF++ Distribution")
axes[0].set_xlabel("chrF++ Score")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# Plot BERTScore F1 distribution if available
if "sentence_BERTScore_F1" in metrics_en_to_ar:
    axes[1].hist(metrics_en_to_ar["sentence_BERTScore_F1"], bins=25, alpha=0.6, label="English -> Darija", color="#1f77b4", edgecolor="black")
    axes[1].hist(metrics_ar_to_en["sentence_BERTScore_F1"], bins=25, alpha=0.6, label="Darija -> English", color="#2ca02c", edgecolor="black")
    axes[1].set_title("Sentence-Level BERTScore F1 Distribution")
    axes[1].set_xlabel("BERTScore F1 (%)")
    axes[1].set_ylabel("Count")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.5)
else:
    axes[1].text(0.5, 0.5, "BERTScore not computed\n(pip install bert-score)", ha="center", va="center", transform=axes[1].transAxes, fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# Qualitative Error & Success Analysis
eval_df["chrf_en_to_ar"] = metrics_en_to_ar["sentence_chrF++"]
eval_df["chrf_ar_to_en"] = metrics_ar_to_en["sentence_chrF++"]

print("\n" + "="*70)
print("Top 3 High-Scoring Translations (English -> Darija):")
print("="*70)
for _, row in eval_df.sort_values(by="chrf_en_to_ar", ascending=False).head(3).iterrows():
    print(f"Source (EN)    : {row['English']}")
    print(f"Reference (AR) : {row['Arabic']}")
    print(f"Prediction (AR): {row['pred_ar']}")
    print(f"chrF++ Score   : {row['chrf_en_to_ar']:.2f}\n")

print("="*70)
print("Top 3 Lowest-Scoring Translations (Failure Modes / Variations):")
print("="*70)
for _, row in eval_df.sort_values(by="chrf_en_to_ar", ascending=True).head(3).iterrows():
    print(f"Source (EN)    : {row['English']}")
    print(f"Reference (AR) : {row['Arabic']}")
    print(f"Prediction (AR): {row['pred_ar']}")
    print(f"chrF++ Score   : {row['chrf_en_to_ar']:.2f}\n")


In [ ]:
# Save Predictions and Metric Results to File
eval_csv_file = "stage2_evaluation_results.csv"
eval_df.to_csv(eval_csv_file, index=False, encoding="utf-8-sig")
print(f"✓ Saved full evaluation table to {eval_csv_file}")

summary_json_file = "stage2_evaluation_summary.json"
metrics_export = {
    "sample_size": len(eval_df),
    "english_to_darija": {k: v for k, v in metrics_en_to_ar.items() if not k.startswith("sentence_")},
    "darija_to_english": {k: v for k, v in metrics_ar_to_en.items() if not k.startswith("sentence_")},
}

with open(summary_json_file, "w", encoding="utf-8") as f:
    json.dump(metrics_export, f, indent=4, ensure_ascii=False)
print(f"✓ Saved summary metrics to {summary_json_file}")
